# Hierarchical Bayesian Accrual Model (sigma)

**Purpose:** Estimate firm-specific accrual noise `sigma_i` using a Hierarchical
Bayesian (HB) model. This is the primary method (vs OLS RMSE in notebook 02).

**Model structure:**
```
Observation:    WCA_{i,t} ~ Normal(mu_{i,t}, sigma²_i)
                mu_{i,t}  = alpha_i + b1*CFO_{t-1} + b2*CFO_t + b3*CFO_{t+1}*
                                    + b4*dREV + b5*PPE
                * CFO_{t+1} = NA for portfolio year t

Firm level:     sigma_i ~ HalfNormal(sigma_j[i])
                alpha_i ~ Normal(alpha_j[i], tau²)

Industry level: sigma_j ~ HalfNormal(sigma_0)
                alpha_j ~ Normal(mu_0, omega²)

Market level:   sigma_0 ~ HalfNormal(0.1)
                mu_0    ~ Normal(0, 1)
                tau, omega ~ HalfNormal(0.1)
```

**Output:** `sigma_hb.parquet` — posterior mean E[sigma_i] and 2000 samples per firm-year.

In [ ]:
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

notebook_dir = Path().resolve()
sys.path.insert(0, str(notebook_dir.parents[0] / "src"))

from config import PROCESSED_DIR

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

# Sampling settings
N_DRAWS     = 2000
N_TUNE      = 1000
N_CHAINS    = 2
RANDOM_SEED = 42

print(f'PyMC version: {pm.__version__}')
print(f'ArviZ version: {az.__version__}')

## 1. Load WCA data

In [ ]:
wca_data = pd.read_parquet(PROCESSED_DIR / "wca_data.parquet")

# Drop rows where any regressor is missing
REGRESSORS = ['WCA_scaled', 'CFO_scaled', 'dREV_scaled', 'PPE_scaled']
wca_clean  = wca_data.dropna(subset=REGRESSORS).copy()

print(f'Firm-years available: {len(wca_clean):,}')
print(f'Tickers: {wca_clean["Ticker"].nunique()}')

# Encode firm and industry as integer indices (required by PyMC)
wca_clean['firm_idx'], firm_labels = pd.factorize(wca_clean['Ticker'])

industry_col = 'Industry' if 'Industry' in wca_clean.columns else 'Sector'
wca_clean['industry_idx'], industry_labels = pd.factorize(wca_clean[industry_col])

# Map firm -> industry (each firm belongs to one industry)
firm_to_industry = (
    wca_clean.groupby('firm_idx')['industry_idx']
    .first()
    .reindex(range(len(firm_labels)))
    .fillna(0)
    .astype(int)
    .values
)

n_firms      = len(firm_labels)
n_industries = len(industry_labels)

print(f'Firms: {n_firms}, Industries: {n_industries}')

## 2. Prepare data for rolling window estimation

For each portfolio year t, we estimate the HB model on training years t-4 to t-1
(expanding to rolling). CFO_{t+1} is observed in training years, NA for year t.

In [ ]:
def prepare_window(df: pd.DataFrame, portfolio_year: int,
                   min_train: int = 3, max_train: int = 5) -> tuple:
    """
    Prepare training and portfolio-year data for one estimation window.

    Training years: t-max_train to t-1 (full McNichols with CFO_{t+1} observed).
    Portfolio year t: CFO_{t+1} = 0 (marginalised — residual absorbs the uncertainty).

    Returns (train_df, port_df) or (None, None) if insufficient data.
    """
    port_df  = df[df['Year'] == portfolio_year].copy()
    train_df = df[(df['Year'] >= portfolio_year - max_train) &
                  (df['Year'] <  portfolio_year)].copy()

    if train_df['Year'].nunique() < min_train or len(port_df) == 0:
        return None, None

    # Build CFO_{t+1} in training data (shift back one year within firm)
    cfo_lead = (
        df[df['Year'] <= portfolio_year]
        .sort_values(['Ticker', 'Year'])
        .groupby('Ticker')['CFO_scaled']
        .shift(-1)
    )
    train_df['CFO_lead1'] = cfo_lead.reindex(train_df.index)

    # Build CFO_{t-1} in training data
    cfo_lag = (
        df.sort_values(['Ticker', 'Year'])
        .groupby('Ticker')['CFO_scaled']
        .shift(1)
    )
    train_df['CFO_lag1'] = cfo_lag.reindex(train_df.index)
    port_df['CFO_lag1']  = cfo_lag.reindex(port_df.index)

    # For portfolio year: CFO_{t+1} = 0 (marginalised)
    port_df['CFO_lead1'] = 0.0

    # Drop training rows with missing CFO lags
    train_df = train_df.dropna(subset=['CFO_lag1', 'CFO_lead1'])
    port_df  = port_df.dropna(subset=['CFO_lag1'])

    return train_df, port_df

## 3. Hierarchical Bayesian model (single window)

We define the HB model as a function so it can be re-run for each portfolio year.

In [ ]:
def build_hb_model(train_df: pd.DataFrame, port_df: pd.DataFrame) -> pm.Model:
    """
    Build PyMC hierarchical Bayesian accrual model.

    Parameters
    ----------
    train_df : training years data (full McNichols)
    port_df  : portfolio year data (CFO_lead1 = 0)

    Returns
    -------
    PyMC model (not yet sampled)
    """
    combined = pd.concat([train_df, port_df], ignore_index=True)

    # Re-encode indices for this window's firms
    combined['firm_idx'], f_labels = pd.factorize(combined['Ticker'])
    combined['industry_idx'], i_labels = pd.factorize(combined['Industry'
                                            if 'Industry' in combined.columns else 'Sector'])
    firm_to_ind = (
        combined.groupby('firm_idx')['industry_idx']
        .first().reindex(range(combined['firm_idx'].nunique()))
        .fillna(0).astype(int).values
    )

    n_f = combined['firm_idx'].nunique()
    n_i = combined['industry_idx'].nunique()

    wca    = combined['WCA_scaled'].values.astype(float)
    cfo_l1 = combined['CFO_lag1'].values.astype(float)
    cfo_t  = combined['CFO_scaled'].values.astype(float)
    cfo_f1 = combined['CFO_lead1'].values.astype(float)
    drev   = combined['dREV_scaled'].values.astype(float)
    ppe    = combined['PPE_scaled'].values.astype(float)
    fi     = combined['firm_idx'].values

    with pm.Model() as model:
        # ── Market-level hyperpriors ──────────────────────────────────────
        sigma_0 = pm.HalfNormal('sigma_0', sigma=0.1)
        mu_0    = pm.Normal('mu_0', mu=0, sigma=1)
        tau     = pm.HalfNormal('tau',   sigma=0.1)
        omega   = pm.HalfNormal('omega', sigma=0.1)

        # ── Industry-level ────────────────────────────────────────────────
        sigma_j = pm.HalfNormal('sigma_j', sigma=sigma_0, shape=n_i)
        alpha_j = pm.Normal('alpha_j', mu=mu_0, sigma=omega, shape=n_i)

        # ── Firm-level ────────────────────────────────────────────────────
        sigma_i = pm.HalfNormal('sigma_i', sigma=sigma_j[firm_to_ind], shape=n_f)
        alpha_i = pm.Normal('alpha_i', mu=alpha_j[firm_to_ind], sigma=tau, shape=n_f)

        # ── Shared regression coefficients ───────────────────────────────
        b1 = pm.Normal('b1', mu=0, sigma=1)   # CFO_{t-1}
        b2 = pm.Normal('b2', mu=0, sigma=1)   # CFO_t
        b3 = pm.Normal('b3', mu=0, sigma=1)   # CFO_{t+1} (0 for port year)
        b4 = pm.Normal('b4', mu=0, sigma=1)   # dREV
        b5 = pm.Normal('b5', mu=0, sigma=1)   # PPE

        # ── Observation model ─────────────────────────────────────────────
        mu = (alpha_i[fi]
              + b1 * cfo_l1
              + b2 * cfo_t
              + b3 * cfo_f1
              + b4 * drev
              + b5 * ppe)

        obs = pm.Normal('obs', mu=mu, sigma=sigma_i[fi], observed=wca)

        # Store firm labels for post-processing
        model.firm_labels    = f_labels
        model.combined       = combined

    return model

## 4. Run rolling estimation

For each portfolio year, sample the HB model and extract posterior sigma_i.

In [ ]:
all_years = sorted(wca_clean['Year'].unique())
# Portfolio years start after min 3 training years
portfolio_years = all_years[3:]

print(f'Training on years: {all_years[:3]} (startup)')
print(f'Portfolio years to estimate: {portfolio_years}')
print(f'Total windows: {len(portfolio_years)}')

In [ ]:
sigma_records = []
traces = {}   # store traces for diagnostics

for port_year in portfolio_years:
    print(f'\nEstimating portfolio year {port_year}...')

    train_df, port_df = prepare_window(wca_clean, port_year)
    if train_df is None:
        print(f'  Skipped — insufficient training data.')
        continue

    print(f'  Training years: {sorted(train_df["Year"].unique())}')
    print(f'  Training firm-years: {len(train_df)}, Portfolio firm-years: {len(port_df)}')

    model = build_hb_model(train_df, port_df)

    with model:
        trace = pm.sample(
            draws       = N_DRAWS,
            tune        = N_TUNE,
            chains      = N_CHAINS,
            random_seed = RANDOM_SEED,
            progressbar = True,
            target_accept = 0.9,
        )

    traces[port_year] = trace

    # Extract sigma_i posterior for each firm in portfolio year
    sigma_samples = trace.posterior['sigma_i'].values  # shape: (chains, draws, n_firms)
    sigma_flat    = sigma_samples.reshape(-1, sigma_samples.shape[-1])  # (chains*draws, n_firms)

    # Map firm indices back to tickers
    combined_window = pd.concat([train_df, port_df], ignore_index=True)
    combined_window['firm_idx_w'], f_labels_w = pd.factorize(combined_window['Ticker'])

    port_tickers = port_df['Ticker'].unique()

    for ticker in port_tickers:
        rows = combined_window[combined_window['Ticker'] == ticker]
        if len(rows) == 0:
            continue
        f_idx = rows['firm_idx_w'].iloc[0]
        if f_idx >= sigma_flat.shape[1]:
            continue

        samples = sigma_flat[:, f_idx]
        sigma_records.append({
            'Ticker'         : ticker,
            'Year'           : port_year,
            'sigma_hb_mean'  : float(np.mean(samples)),
            'sigma_hb_std'   : float(np.std(samples)),
            'sigma_hb_p5'    : float(np.percentile(samples, 5)),
            'sigma_hb_p95'   : float(np.percentile(samples, 95)),
            'sigma_hb_samples': samples.tolist(),  # all 2000*chains samples
        })

    print(f'  Done. Extracted sigma for {len(port_tickers)} firms.')

print(f'\nTotal firm-years with HB sigma: {len(sigma_records)}')

## 5. MCMC diagnostics

Check R-hat and effective sample size for the most recent estimation window.

In [ ]:
if traces:
    last_year  = max(traces.keys())
    last_trace = traces[last_year]

    print(f'Diagnostics for portfolio year {last_year}:')
    summary = az.summary(last_trace, var_names=['sigma_0', 'mu_0', 'b1', 'b2', 'b3', 'b4', 'b5'])
    print(summary[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']].round(3))

    rhat_max = summary['r_hat'].max()
    if rhat_max > 1.05:
        print(f'\nWARNING: max R-hat = {rhat_max:.3f} > 1.05 — consider more tuning steps.')
    else:
        print(f'\nR-hat OK: max = {rhat_max:.3f}')

## 6. Save outputs

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

sigma_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'sigma_hb_samples'}
                          for r in sigma_records])

# Save point summaries as parquet
sigma_df.to_parquet('../data/processed/sigma_hb.parquet', index=False)

# Save full samples as pickle (needed for Method 3 / full propagation)
import pickle
with open('../data/processed/sigma_hb_samples.pkl', 'wb') as f:
    pickle.dump(sigma_records, f)

print('Saved:')
print('  data/processed/sigma_hb.parquet       (point summaries)')
print('  data/processed/sigma_hb_samples.pkl   (full posterior samples)')
print(f'  {len(sigma_df):,} firm-years')
sigma_df.describe().round(4)